# ECS 170 Stage 4 Part D

This notebook reproduces experiments for both tasks: 18 generation runs (three cell types, six settings) and three classification GRU runs (baseline, hidden size 256, learning rate 0.0005). Select **Runtime > Change runtime type > T4 GPU**, then run the cells. Results save to Google Drive after each model. The notebook is prepared for CUDA; the packaged measured results were obtained on CPU.

Classification uses fine-tuned GloVe 100d, three epochs, 22,500 training reviews, 2,500 validation reviews, and the original 25,000-review test set. Compare its variations against the new Part D baseline. Part B trained on all 25,000 training reviews, so its supplied scores are a separate reference.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
import sys, subprocess, zipfile, shutil
import torch
print("PyTorch:", torch.__version__, "CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print("CPU runtime selected; classification will take longer.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy", "matplotlib", "scikit-learn", "gdown"], check=True)


In [ ]:
REPO = Path("/content/ecs170-project")
if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/katherinyuh/ecs170-project.git", str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "ef719fea7ce843f6079128d56ef141a43cfec5ec"], check=True)
RUNNER = REPO / "script/stage_4_script/run_part_d.py"
RUNNER.write_text('"""Part D: change one factor at a time; save real metrics, curves and stories.\n\nFrom repository root:\n  python script/stage_4_script/run_part_d.py --task generation\n  python script/stage_4_script/run_part_d.py --task classification --epochs 3\n  python script/stage_4_script/run_part_d.py --task both --quick\n  python script/stage_4_script/run_part_d.py --plan\n\nClassification embeddings are learned unless --glove is supplied. A pretrained\nembedding dimension sweep requires separate matching GloVe files, so the runner\nrefuses that sweep with --glove rather than silently changing initialization.\n"""\nimport argparse\nimport csv\nimport gc\nimport hashlib\nimport json\nimport sys\nimport time\nfrom pathlib import Path\n\nif hasattr(sys.stdout, \'reconfigure\'):\n    sys.stdout.reconfigure(line_buffering=True)\n\nROOT = Path(__file__).resolve().parents[2]\nsys.path.insert(0, str(ROOT))\nVARIANTS = {\n    \'baseline\': {},\n    \'embedding50\': {\'embedding_dim\': 50},\n    \'hidden256\': {\'hidden_size\': 256},\n    \'layers2\': {\'num_layers\': 2},\n    \'sequence\': {},  # generation context 3 -> 5; classification review cap 200 -> 300\n    \'lr0005\': {\'learning_rate\': 0.0005},\n}\nPROMPTS = [\'what did the\', \'why did the\', \'two artists had\',\n           \'my invisible penguin\', \'zebra astronauts juggle\']\n\n\ndef configurations(task, epochs=None):\n    base = dict(embedding_dim=100, hidden_size=128, num_layers=1,\n                dropout=0.2 if task == \'generation\' else 0.3,\n                max_epoch=epochs or (15 if task == \'generation\' else 10),\n                learning_rate=0.001, batch_size=64, grad_clip=5.0,\n                optimizer_name=\'adam\', seed=2)\n    if task == \'generation\':\n        base[\'seq_len\'] = 3\n    else:\n        base.update(bidirectional=False, track_test_acc=True)\n    answer = {}\n    for name, change in VARIANTS.items():\n        cfg = {**base, **change}\n        if name == \'sequence\' and task == \'generation\':\n            cfg[\'seq_len\'] = 5\n        answer[name] = (cfg, 300 if name == \'sequence\' and task == \'classification\' else 200)\n    return answer\n\n\ndef write_json(path, data):\n    temporary = path.with_suffix(path.suffix + \'.tmp\')\n    temporary.write_text(json.dumps(data, indent=2) + \'\\n\')\n    temporary.replace(path)\n\n\ndef sha256_file(path):\n    digest = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda: stream.read(8 * 1024 * 1024), b\'\'):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef digest_data(data):\n    h = hashlib.sha256()\n    for split in (\'train\', \'test\'):\n        if split not in data:\n            continue\n        for key in (\'X\', \'y\'):\n            a = data[split][key]\n            h.update(str(a.shape).encode())\n            h.update(a.tobytes())\n    h.update(json.dumps(data[\'vocab\'][\'id_to_word\']).encode())\n    return h.hexdigest()\n\n\ndef plot_history(record, path):\n    import matplotlib\n    matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\n    h = record[\'history\']\n    x = range(1, len(h[\'loss\']) + 1)\n    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))\n    axes[0].plot(x, h[\'loss\'], label=\'training\')\n    axes[0].set(xlabel=\'Epoch\', ylabel=\'Cross entropy\', title=\'Training loss\')\n    axes[1].plot(x, h[\'train_acc\'], label=\'training, dropout on\')\n    if h.get(\'validation_acc\'):\n        axes[1].plot(x, h[\'validation_acc\'], label=\'validation, dropout off\')\n    axes[1].set(xlabel=\'Epoch\', ylabel=\'Accuracy\', title=\'Next token\' if record[\'task\'] == \'generation\' else \'Sentiment\')\n    for ax in axes:\n        ax.grid(alpha=0.25)\n        ax.legend(fontsize=8)\n    fig.suptitle(record[\'run_id\'])\n    fig.tight_layout()\n    fig.savefig(path, dpi=160)\n    plt.close(fig)\n\n\ndef refresh_summary(out):\n    records = [json.loads(p.read_text()) for p in sorted(out.glob(\'*/*/record.json\'))]\n    if not records:\n        return\n    fields = [\'task\',\'cell_type\',\'variation\',\'embedding_dim\',\'hidden_size\',\'num_layers\',\n              \'seq_len\',\'max_len\',\'learning_rate\',\'max_epoch\',\'dropout\',\'batch_size\',\n              \'seed\',\'embedding_source\',\'recurrent_dropout\',\'device\',\'training_examples\',\n              \'params\',\'minutes\',\'final_train_loss\',\'final_train_acc\',\n              \'train_eval_acc\',\'validation_accuracy\',\'accuracy\',\'precision_macro\',\'recall_macro\',\'f1_macro\']\n    with (out / \'part_d_results.csv\').open(\'w\', newline=\'\') as f:\n        writer = csv.DictWriter(f, fieldnames=fields, extrasaction=\'ignore\')\n        writer.writeheader()\n        for r in records:\n            writer.writerow({**r, **r[\'config\'], **r[\'metrics\']})\n    write_json(out / \'part_d_history.json\', {r[\'run_id\']: r for r in records})\n    import matplotlib\n    matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\n    for task in sorted({r[\'task\'] for r in records}):\n        present_cells = [cell for cell in (\'rnn\', \'lstm\', \'gru\')\n                         if any(r[\'task\'] == task and r[\'cell_type\'] == cell for r in records)]\n        fig, axes = plt.subplots(1, len(present_cells), figsize=(4.5 * len(present_cells), 3.5), squeeze=False)\n        for ax, cell in zip(axes[0], present_cells):\n            for r in records:\n                if r[\'task\'] == task and r[\'cell_type\'] == cell:\n                    h = r[\'history\'][\'loss\']\n                    ax.plot(range(1, len(h)+1), h, label=r[\'variation\'])\n            ax.set(title=cell.upper(), xlabel=\'Epoch\', ylabel=\'Training cross entropy\')\n            ax.grid(alpha=0.25)\n            if ax.lines:\n                ax.legend(fontsize=7)\n        fig.tight_layout()\n        fig.savefig(out / f\'{task}_ablation_curves.png\', dpi=160)\n        plt.close(fig)\n\n\ndef run_task(task, args, out):\n    import numpy as np\n    import torch\n    from sklearn.model_selection import train_test_split\n    from local_code.stage_2_code.Evaluate_Metrics import Evaluate_Metrics\n    from local_code.stage_4_code.Dataset_Loader_Classification import Dataset_Loader_Classification\n    from local_code.stage_4_code.Dataset_Loader_Generation import Dataset_Loader_Generation\n    from local_code.stage_4_code.Method_RNN_Classification import Method_RNN_Classification\n    from local_code.stage_4_code.Method_RNN_Generation import Method_RNN_Generation\n    from local_code.stage_4_code.Glove_Utils import load_glove_matrix\n    from run_generation_experiments import prepare_joke_index, compare_story\n\n    cache, glove_cache = {}, {}\n    for variation, (base_cfg, max_len) in configurations(task, args.epochs).items():\n        if variation not in args.variants:\n            continue\n        cfg = {**base_cfg, \'device_name\': args.device}\n        if args.quick:\n            cfg.update(max_epoch=1, train_subset=256)\n        cache_key = cfg.get(\'seq_len\', max_len)\n        if cache_key not in cache:\n            loader = Dataset_Loader_Generation() if task == \'generation\' else Dataset_Loader_Classification()\n            if task == \'generation\':\n                loader.context_len = cfg[\'seq_len\']\n            else:\n                loader.max_len = max_len\n            if args.data_dir:\n                loader.dataset_source_folder_path = Path(args.data_dir) / (\'text_generation/data\' if task == \'generation\' else \'text_classification\')\n            data = loader.load()\n            data_hash = digest_data(data)\n            original_test = None\n            if task == \'classification\':\n                original_test = data[\'test\']\n                ids = np.arange(len(data[\'train\'][\'y\']))\n                train_ids, val_ids = train_test_split(ids, test_size=0.1, random_state=2, stratify=data[\'train\'][\'y\'])\n                original_train = data[\'train\']\n                data = {**data, \'train\': {k: v[train_ids] for k,v in original_train.items()},\n                        \'test\': {k: v[val_ids] for k,v in original_train.items()}}\n                if args.quick:\n                    # Smoke checks exercise scoring on small, stratified subsets.\n                    for split in (\'validation\', \'test\'):\n                        source = data[\'test\'] if split == \'validation\' else original_test\n                        if len(source[\'y\']) > 256:\n                            keep, _ = train_test_split(np.arange(len(source[\'y\'])), train_size=256,\n                                                       random_state=2, stratify=source[\'y\'])\n                            limited = {k: v[keep] for k, v in source.items()}\n                            if split == \'validation\':\n                                data[\'test\'] = limited\n                            else:\n                                original_test = limited\n            cache[cache_key] = (loader, data, data_hash, original_test)\n        loader, data, data_hash, original_test = cache[cache_key]\n        embedding_source = \'learned from scratch\'\n        glove = None\n        if task == \'classification\' and args.glove:\n            if cache_key not in glove_cache:\n                glove_cache[cache_key] = load_glove_matrix(args.glove, data[\'vocab\'][\'id_to_word\'])\n            glove = glove_cache[cache_key]\n            if glove[0].shape[1] != cfg[\'embedding_dim\']:\n                raise ValueError(\'GloVe dimension must equal --embedding-dim baseline of 100. Use matching 100d vectors, or omit --glove.\')\n            embedding_source = f\'GloVe {glove[0].shape[1]}d fine tuned\'\n        for cell in args.cells:\n            config = {**cfg, \'cell_type\': cell}\n            run_id = f\'{task}_{cell}_{variation}\'\n            folder = out / task / f\'{cell}_{variation}\'\n            identity = {\'config\': config, \'max_len\': max_len if task == \'classification\' else None,\n                        \'data_sha256\': data_hash, \'embedding_source\': embedding_source,\n                        \'glove_sha256\': args.glove_sha256 if task == \'classification\' and args.glove else None,\n                        \'validation_fraction\': 0.1 if task == \'classification\' else None,\n                        \'recurrent_dropout\': 0.0}\n            if args.quick:\n                identity[\'quick_eval_cap\'] = 256\n            if (folder / \'record.json\').exists():\n                saved = json.loads((folder / \'record.json\').read_text())\n                if saved[\'identity\'] != identity:\n                    raise ValueError(f\'{run_id}: saved settings/data differ; choose a new --out directory.\')\n                print(\'Skipping completed\', run_id, flush=True)\n                continue\n            folder.mkdir(parents=True, exist_ok=True)\n            print(\'\\nRUN\', run_id, json.dumps(config), flush=True)\n            method_class = Method_RNN_Generation if task == \'generation\' else Method_RNN_Classification\n            model = method_class(run_id, \'Part D single factor experiment\', config=config)\n            model.data = data\n            if glove is not None:\n                model.embedding_matrix = glove[0]\n            # Keep recurrent dropout zero at BOTH depths. Embedding/output dropout stays fixed.\n            model.build_model(data[\'vocab\'][\'size\'])\n            model.rnn.dropout = 0.0\n            if model.device.type == \'cuda\':\n                torch.cuda.synchronize()\n            start = time.perf_counter()\n            result = model.run()\n            metrics = {}\n            if task == \'generation\':\n                metrics[\'train_eval_acc\'] = model.final_eval_acc\n                index = prepare_joke_index(data[\'jokes\'])\n                stories = []\n                for prompt in PROMPTS:\n                    for temperature in (None, 0.8):\n                        generated = model.generate(prompt, temperature=temperature)\n                        stories.append({\'prompt\': prompt, \'temperature\': temperature,\n                                        \'text\': loader.ids_to_text(generated[\'ids\']),\n                                        \'unknown_words\': generated[\'unknown_words\'],\n                                        \'stopped_on\': generated[\'stopped_on\'],\n                                        \'comparison\': compare_story(generated[\'words\'], len(generated[\'prompt_words\']), index)})\n                write_json(folder / \'stories.json\', stories)\n            else:\n                evaluator = Evaluate_Metrics(\'Part D evaluation\', \'\')\n                evaluator.data = result\n                metrics[\'validation_accuracy\'] = float(evaluator.evaluate()[\'accuracy\'])\n                # The original test set is scored once, after the fixed final epoch.\n                # Never use it for per-epoch curves or choosing a checkpoint.\n                pred = model.test(original_test[\'X\'])\n                evaluator.data = {\'true_y\': original_test[\'y\'], \'pred_y\': pred}\n                metrics.update({k: float(v) for k,v in evaluator.evaluate().items()})\n                np.savez_compressed(folder / \'predictions.npz\', true_y=original_test[\'y\'], pred_y=pred)\n            if model.device.type == \'cuda\':\n                torch.cuda.synchronize()\n            record = {\'run_id\': run_id, \'task\': task, \'cell_type\': cell, \'variation\': variation,\n                      \'config\': config, \'identity\': identity, \'max_len\': identity[\'max_len\'],\n                      \'embedding_source\': embedding_source, \'recurrent_dropout\': 0.0,\n                      \'device\': str(model.device), \'torch_version\': torch.__version__,\n                      \'validation_examples\': len(data[\'test\'][\'y\']) if task == \'classification\' else 0,\n                      \'test_examples\': len(original_test[\'y\']) if task == \'classification\' else 0,\n                      \'glove_stats\': glove[1] if glove is not None else None,\n                      \'training_examples\': min(len(data[\'train\'][\'y\']), config.get(\'train_subset\') or len(data[\'train\'][\'y\'])),\n                      \'params\': sum(p.numel() for p in model.parameters()),\n                      \'minutes\': (time.perf_counter()-start)/60,\n                      \'final_train_loss\': model.loss_history[-1], \'final_train_acc\': model.train_acc_history[-1],\n                      \'metrics\': metrics, \'history\': {\'loss\': model.loss_history,\n                      \'train_acc\': model.train_acc_history, \'validation_acc\': getattr(model, \'test_acc_history\', []),\n                      \'epoch_seconds\': model.epoch_times}}\n            if args.save_weights:\n                torch.save({\'state_dict\': model.state_dict(), \'config\': config, \'vocab\': data[\'vocab\'],\n                            \'identity\': identity}, folder / \'weights.pt\')\n            plot_history(record, folder / \'learning_curve.png\')\n            write_json(folder / \'record.json\', record)\n            refresh_summary(out)\n            del model\n            gc.collect()\n            if torch.cuda.is_available():\n                torch.cuda.empty_cache()\n\n\ndef main():\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--task\', choices=[\'generation\',\'classification\',\'both\'], default=\'generation\')\n    p.add_argument(\'--cells\', nargs=\'+\', choices=[\'rnn\',\'lstm\',\'gru\'], default=[\'rnn\',\'lstm\',\'gru\'])\n    p.add_argument(\'--variants\', nargs=\'+\', choices=list(VARIANTS), default=list(VARIANTS))\n    p.add_argument(\'--epochs\', type=int)\n    p.add_argument(\'--data-dir\', help=\'path containing text_classification/ and text_generation/\')\n    p.add_argument(\'--out\', type=Path)\n    p.add_argument(\'--device\', default=\'auto\', choices=[\'auto\',\'cpu\',\'cuda\',\'mps\'])\n    p.add_argument(\'--threads\', type=int, default=2, help=\'CPU threads; avoid excessive thread overhead on short sequences\')\n    p.add_argument(\'--glove\', help=\'optional 100d GloVe for classification; omit embedding50 variant\')\n    p.add_argument(\'--save-weights\', action=\'store_true\')\n    p.add_argument(\'--quick\', action=\'store_true\', help=\'one epoch on 256 training examples; smoke check only\')\n    p.add_argument(\'--plan\', action=\'store_true\', help=\'print experiment settings without loading data\')\n    args = p.parse_args()\n    tasks = [\'generation\',\'classification\'] if args.task == \'both\' else [args.task]\n    if args.epochs is not None and args.epochs < 1:\n        p.error(\'--epochs must be positive\')\n    if args.threads < 1:\n        p.error(\'--threads must be positive\')\n    if args.glove and \'classification\' in tasks and \'embedding50\' in args.variants:\n        p.error(\'With GloVe select --variants baseline hidden256 layers2 sequence lr0005. Compare embedding sizes separately with matched initialization.\')\n    if args.plan:\n        print(json.dumps({t: {k:{\'model\': c,\'max_len\': m if t==\'classification\' else None}\n              for k,(c,m) in configurations(t,args.epochs).items() if k in args.variants} for t in tasks}, indent=2))\n        return\n    import torch\n    torch.set_num_threads(args.threads)\n    args.glove_sha256 = sha256_file(args.glove) if args.glove else None\n    out = args.out or ROOT / \'result/stage_4_result\' / (\'part_d_quick\' if args.quick else \'part_d\')\n    out.mkdir(parents=True, exist_ok=True)\n    for task in tasks:\n        run_task(task, args, out)\n    refresh_summary(out)\n    print(\'Saved:\', out.resolve())\n\n\nif __name__ == \'__main__\':\n    main()\n', encoding="utf-8")


## Select the experiment set

The default matches the completed Part D experiment plan. Use `QUICK=True` only for an installation check; it trains on 256 examples for one epoch and does not produce report results. All changes are relative to the task baseline, one setting at a time. Generation baseline: embedding 100, hidden 128, depth 1, context 3, learning rate 0.001. Classification baseline: GloVe 100d, hidden 128, depth 1, review length 200, learning rate 0.001.

In [ ]:
TASK = "both"  # "both", "generation", or "classification"
QUICK = False
SAVE_WEIGHTS = False
GENERATION_CELLS = ["rnn", "lstm", "gru"]
GENERATION_VARIANTS = ["baseline", "embedding50", "hidden256", "layers2", "sequence", "lr0005"]
CLASSIFICATION_CELLS = ["gru"]
CLASSIFICATION_VARIANTS = ["baseline", "hidden256", "lr0005"]
GENERATION_EPOCHS = 15
CLASSIFICATION_EPOCHS = 3
OUTPUT_ROOT = Path("/content/drive/MyDrive/ecs170-stage4-part-d-both")
GLOVE = REPO / "data/stage_4_data/glove/glove_subset_100d.txt"


## Instructor dataset

The archive is about 66 MB. If Drive download quotas block it, download it from the assignment and place it at `/content/stage_4_data.zip`, then rerun this cell.

In [ ]:
import gdown
archive = Path("/content/stage_4_data.zip")
if not archive.exists():
    gdown.download(id="1yNr8deq1JO-HNhlAa1yV4Zycjqmr1Hfz", output=str(archive), quiet=False)
if not archive.is_file() or not zipfile.is_zipfile(archive):
    raise RuntimeError("Dataset download failed. Place the assignment ZIP in /content and retry.")
destination = REPO / "data"
destination.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as z:
    for name in z.namelist():
        if name.startswith("stage_4_data/") and "__MACOSX" not in name and not name.endswith(".DS_Store"):
            if not (destination / name).resolve().is_relative_to(destination.resolve()):
                raise ValueError("Unsafe ZIP entry")
            z.extract(name, destination)
print("Dataset ready")


## Matching classification embeddings

Generation learns its own embeddings and skips this step. For classification, this cell downloads the instructor-linked 100d GloVe archive and exports only the IMDB vocabulary. To avoid the 555 MB download, upload `glove_subset_100d.txt` from your team’s Part B files to `/content/` first. The original 50d vectors cannot reproduce the 100d baseline. The subset should contain 19,345 vectors.

In [ ]:
if TASK in ("both", "classification"):
    GLOVE.parent.mkdir(parents=True, exist_ok=True)
    local_subset = Path("/content/glove_subset_100d.txt")
    if not GLOVE.exists() and local_subset.exists():
        shutil.copyfile(local_subset, GLOVE)
    if not GLOVE.exists():
        glove_archive = Path("/content/glove100d.zip")
        if not glove_archive.exists():
            gdown.download(id="10F62gyUWe3EPHBhI8L7_NsO5Gp0PB3NR", output=str(glove_archive), quiet=False)
        if not glove_archive.is_file() or not zipfile.is_zipfile(glove_archive):
            raise RuntimeError("GloVe download failed. Upload the matching 100d subset and retry.")
        if str(REPO) not in sys.path:
            sys.path.insert(0, str(REPO))
        from local_code.stage_4_code.Dataset_Loader_Classification import Dataset_Loader_Classification
        from local_code.stage_4_code.Glove_Utils import _lines
        vocab = set(Dataset_Loader_Classification().load()["vocab"]["id_to_word"])
        temporary = GLOVE.with_suffix(".tmp")
        count = 0
        with temporary.open("w", encoding="utf-8") as stream:
            for line in _lines(glove_archive):
                word = line.split(" ", 1)[0]
                if word in vocab and len(line.split()) == 101:
                    stream.write(line.rstrip() + chr(10))
                    vocab.discard(word)
                    count += 1
        if count != 19345:
            temporary.unlink(missing_ok=True)
            raise RuntimeError(f"Expected 19345 vectors, found {count}. Check the GloVe version.")
        temporary.replace(GLOVE)
    print("GloVe subset:", GLOVE, "bytes:", GLOVE.stat().st_size)


## Run the experiments

The classification test set is scored only after the fixed final epoch; per-epoch curves use validation data. Every completed run has a CSV entry, JSON history and learning curve. Generation also saves prompt outputs and overlap checks; classification saves test predictions. Matching completed runs are skipped. Use a new output folder when changing settings or data. GPU and CPU numerical results can differ.

In [ ]:
tasks = ["generation", "classification"] if TASK == "both" else [TASK]
for task in tasks:
    generation = task == "generation"
    epochs = GENERATION_EPOCHS if generation else CLASSIFICATION_EPOCHS
    cells_for_task = GENERATION_CELLS if generation else CLASSIFICATION_CELLS
    variants = GENERATION_VARIANTS if generation else CLASSIFICATION_VARIANTS
    source = "learned" if generation else "glove100"
    out = OUTPUT_ROOT / (("quick_" if QUICK else "") + f"{task}_{epochs}epochs_{source}")
    cmd = [sys.executable, str(RUNNER), "--task", task, "--epochs", str(epochs),
           "--cells", *cells_for_task, "--variants", *variants, "--out", str(out),
           "--device", "cuda" if torch.cuda.is_available() else "cpu"]
    if not generation:
        cmd += ["--glove", str(GLOVE)]
    if QUICK:
        cmd.append("--quick")
    if SAVE_WEIGHTS:
        cmd.append("--save-weights")
    subprocess.run(cmd, cwd=REPO, check=True)


## Export results

Classification metrics are measured on held-out test reviews. Generation `train_eval_acc` is next-token accuracy on training windows with dropout off; it is not test accuracy or a measure of story quality. The packaged report keeps these measures separate. Timings are descriptive; no GPU speedup is claimed without an actual matched CPU/GPU comparison.

In [ ]:
import pandas as pd
from IPython.display import display
for csv_path in OUTPUT_ROOT.rglob("part_d_results.csv"):
    print(csv_path)
    display(pd.read_csv(csv_path))
export = shutil.make_archive("/content/Stage4_PartD_Colab_results", "zip", OUTPUT_ROOT)
from google.colab import files
files.download(export)
